# 01 - Joint training (ordinary ML reference)

**This is not a continual-learning method.** All ten experiences' training samples are pooled and a
polynomial logistic-regression model is trained once on the pool. It shows what classification quality is reachable when
every experience is available together, using the same model class, optimizer and evaluation sets as notebook 02.
It is *not* a mathematical upper bound: gradient descent with a finite budget may not find the best parameters.

Note: this studies *explicit* polynomial feature expansion (related to polynomial kernels). It is not an implicit-kernel SVM.

## Configuration

In [ ]:
SEED = 0  # random seed for data generation and shuffling
N_TRAIN = 200  # training samples per experience
N_TEST = 500  # fixed evaluation samples per experience (never used for training)
DEGREE = 10  # polynomial degree
LR = 0.5  # learning rate
EPOCHS = 5000  # full-batch gradient-descent steps
NOISE = 0.0  # probability of flipping a TRAINING label (evaluation labels stay clean)

In [ ]:
import importlib.util
import pathlib
import sys

if importlib.util.find_spec("polynomial_cl") is None:  # not pip-installed: use src/
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 80})
from polynomial_cl import (
    PolynomialLogisticModel,
    combine_splits,
    make_experiences,
    metrics,
    train,
)
from polynomial_cl import plotting as P

## Data: ten experiences, ground-truth regions, training samples

In [ ]:
exps = make_experiences(seed=SEED, n_train=N_TRAIN, n_test=N_TEST, noise=NOISE)
x_tr, y_tr, src = combine_splits([e.train for e in exps])
print("pooled training samples:", x_tr.shape, "| class balance:", y_tr.mean().round(3))
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
P.plot_ground_truth(axs[0], title="Ground-truth regions")
P.plot_ground_truth(
    axs[1], samples=(x_tr, y_tr), title="Ground truth + all training samples"
)
plt.tight_layout()
plt.show()

## Train

In [ ]:
res = train(
    PolynomialLogisticModel(DEGREE), x_tr, y_tr, lr=LR, epochs=EPOCHS, seed=SEED
)
model = res.model
print(
    f"loss: {res.loss_history[0]:.4f} -> {res.loss_history[-1]:.4f}   ({model.n_features} coefficients)"
)
P.plot_loss_curve(res.loss_history)
plt.show()

## Evaluation on every experience's fixed test set

In [ ]:
rows = []
for e in exps:
    pred = model.predict(e.test.x)
    rows.append({"experience": e.exp_id, "accuracy": metrics.accuracy(e.test.y, pred)})
rows.append(
    {"experience": "mean", "accuracy": float(np.mean([r["accuracy"] for r in rows]))}
)
print(metrics.format_table(rows))

## Predicted regions, decision boundary, correct / incorrect

In [ ]:
x_te = np.concatenate([e.test.x for e in exps])
y_te = np.concatenate([e.test.y for e in exps])
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
P.plot_ground_truth(axs[0])
P.plot_predictions(
    model,
    axs[1],
    title=f"Predicted regions + boundary f(x)=0 (degree {DEGREE})\n(dashed grey = true boundary)",
)
P.plot_correctness(
    model,
    axs[2],
    samples=(x_te, y_te),
    title="Correct / incorrect regions + evaluation samples\n(x = misclassified evaluation sample)",
)
plt.tight_layout()
plt.show()
print(
    "misclassified evaluation samples:",
    int((model.predict(x_te) != y_te).sum()),
    "of",
    len(y_te),
)

## Learned coefficients

In [ ]:
P.plot_coefficients(model=model, title=f"Final coefficients (degree {DEGREE})")
plt.show()
for n, w in model.coefficients().items():
    print(f"{n:>8s} {w:+10.3f}")

## Optional: effect of the polynomial degree (same data, learning rate and epochs)

Higher degrees add coefficients that start at zero and must be learned by gradient descent; with a fixed step budget
the monomials of high degree are slow to fit, so this is a statement about this optimiser, not about the model class.

In [ ]:
for d in (1, 2, 3, 4, 6, 8, 10):
    m = train(
        PolynomialLogisticModel(d), x_tr, y_tr, lr=LR, epochs=EPOCHS, seed=SEED
    ).model
    accs = [metrics.accuracy(e.test.y, m.predict(e.test.x)) for e in exps]
    print(
        f"degree {d:2d}: mean evaluation accuracy {np.mean(accs):.3f}   min over experiences {np.min(accs):.3f}"
    )